### Librerias

In [32]:
import pandas as pd
import openpyxl
from pathlib import Path

### Concatenacion y Procesamiento en Series

In [ ]:
# ============================
# Configuración
# ============================

folder = Path(r"data\XM\Precio Bolsa Nacional")

years = range(2000, 2025)

daily_series = []
monthly_series = []

# ============================
# Procesamiento
# ============================

for year in years:

    file = folder / f"Precio_Bolsa_Nacional_($kwh)_{year}.xlsx"

    #print(f"Procesando {year}...")

    # Leer archivo
    df = pd.read_excel(
    file,
    header=2,
    usecols="A:Y"
    )

    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

    # Eliminar filas sin fecha
    df.dropna(subset=["Fecha"], inplace=True)

    # Quedarse únicamente con el año correspondiente
    df = df[
        (df["Fecha"] >= f"{year}-01-01") &
        (df["Fecha"] <= f"{year}-12-31")
    ]

    # Reiniciar índice
    df.reset_index(drop=True, inplace=True)

    # Detectar automáticamente si las columnas son enteros o strings
    if 0 in df.columns:
        hour_cols = list(range(24))
    else:
        hour_cols = [str(i) for i in range(24)]

    # Promedio diario
    df["PriceEnergy"] = df[hour_cols].mean(axis=1)

    # Serie diaria
    daily = (
        df[["Fecha", "PriceEnergy"]]
        .rename(columns={"Fecha": "date"})
    )

    # Serie mensual
    monthly = (
        daily
        .set_index("date")
        .resample("MS")
        .mean()
        .reset_index()
    )

    daily_series.append(daily)
    monthly_series.append(monthly)

# ============================
# Unir todas las series
# ============================

price_energy_daily = (
    pd.concat(daily_series, ignore_index=True)
      .sort_values("date")
      .reset_index(drop=True)
)

price_energy_monthly = (
    pd.concat(monthly_series, ignore_index=True)
      .sort_values("date")
      .reset_index(drop=True)
)



### Verificacion:

In [79]:
price_energy_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 9132 entries, 0 to 9131
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         9132 non-null   datetime64[us]
 1   PriceEnergy  9132 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 142.8 KB


In [74]:
price_energy_daily.describe()


,date,PriceEnergy
count,9132,9132.000000
mean,2012-07-01 12:00:00,177.541436
min,2000-01-01 00:00:00,28.841424
25%,2006-04-01 18:00:00,68.651389
50%,2012-07-01 12:00:00,106.911635
75%,2018-10-01 06:00:00,185.624501
max,2024-12-31 00:00:00,2498.800000
std,NaN,211.665528


In [75]:
price_energy_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         300 non-null    datetime64[us]
 1   PriceEnergy  300 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 4.8 KB


In [76]:
price_energy_monthly.describe()

,date,PriceEnergy
count,300,300.000000
mean,2012-06-16 02:14:24,177.633606
min,2000-01-01 00:00:00,35.054972
25%,2006-03-24 06:00:00,70.606008
50%,2012-06-16 00:00:00,107.220757
75%,2018-09-08 12:00:00,188.422075
max,2024-12-01 00:00:00,1529.134288
std,NaN,196.092882


In [78]:
price_energy_daily.head()

,date,PriceEnergy
0,2000-01-01,32.868121
1,2000-01-02,33.034788
2,2000-01-03,37.272288
3,2000-01-04,41.688955
4,2000-01-05,40.893121


In [77]:
price_energy_monthly.head()

,date,PriceEnergy
0,2000-01-01,36.778780
1,2000-02-01,40.261477
2,2000-03-01,37.637297
3,2000-04-01,44.298057
4,2000-05-01,37.291122


### Guardado de las series

In [80]:
output_folder = Path(r"data\\XM\Procesadas")

output_folder.mkdir(parents=True, exist_ok=True)

price_energy_daily.to_csv(
    output_folder / "price_energy_daily_2000-2024.csv",
    index=False
)

price_energy_monthly.to_csv(
    output_folder / "price_energy_monthly_2000-2024.csv",
    index=False
)